In [1]:
pip install -U pandas requests google-play-scraper

In [2]:
import time
import requests
import pandas as pd
import os
from google_play_scraper import reviews, Sort

In [17]:
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "Dataset/data_new_approach")

os.makedirs(DATA_DIR, exist_ok=True)
print(f" Saving files to: {DATA_DIR}")


 Saving files to: /content/Dataset/data_new_approach


In [13]:
APPS = [
    {
        "bank": "Chase",
        "ios_app_id": 298867247,
        "gp_package": "com.chase.sig.android",
    },
    {
        "bank": "Bank of America",
        "ios_app_id": 284847138,
        "gp_package": "com.infonow.bofa",
    },
    {
        "bank": "Wells Fargo",
        "ios_app_id": 311548709,
        "gp_package": "com.wf.wellsfargomobile",
    },
    {
        "bank": "Citi",
        "ios_app_id": 301724680,
        "gp_package": "com.citi.citimobile",
    },
    {
        "bank": "Marcus by Goldman Sachs",
        "ios_app_id": 1489511701,
        "gp_package": "com.marcus.android",
    },
]

In [14]:
def fetch_app_store_reviews_us(app_id: int, max_pages: int = 10, sleep_s: float = 0.8):
    s = requests.Session()
    s.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json,text/plain,*/*",
        "Accept-Language": "en-US,en;q=0.9",
    })

    rows = []

    for page in range(1, max_pages + 1):
        url = f"https://itunes.apple.com/us/rss/customerreviews/page={page}/id={app_id}/sortby=mostrecent/json"
        r = s.get(url, timeout=30)

        ct = (r.headers.get("content-type") or "").lower()
        if r.status_code != 200:
            print(f"[Apple] app_id={app_id} stopped at page={page}, status={r.status_code}")
            break

        if "json" not in ct and "javascript" not in ct:
            print(f"[Apple] app_id={app_id} invalid content at page={page}")
            break

        data = r.json()
        entries = data.get("feed", {}).get("entry", [])

        for e in entries[1:]:  # skip app metadata
            rows.append({
                "platform": "app_store",
                "storefront": "us",
                "app_id": app_id,
                "review_id": e.get("id", {}).get("label"),
                "date": e.get("updated", {}).get("label"),
                "user": e.get("author", {}).get("name", {}).get("label"),
                "rating": int(e.get("im:rating", {}).get("label", 0)),
                "title": e.get("title", {}).get("label"),
                "review": e.get("content", {}).get("label"),
                "version": e.get("im:version", {}).get("label"),
            })

        time.sleep(sleep_s)

    return pd.DataFrame(rows)

In [15]:
def fetch_google_play_reviews_us(package: str, target: int = 500, sleep_s: float = 0.4):
    all_rows = []
    token = None

    while len(all_rows) < target:
        batch = min(200, target - len(all_rows))

        result, token = reviews(
            package,
            lang="en",
            country="us",
            sort=Sort.NEWEST,
            count=batch,
            continuation_token=token,
        )

        if not result:
            break

        for r in result:
            all_rows.append({
                "platform": "google_play",
                "storefront": "us",
                "package": package,
                "review_id": r.get("reviewId"),
                "date": r.get("at"),
                "user": r.get("userName"),
                "rating": r.get("score"),
                "title": r.get("title"),
                "review": r.get("content"),
                "thumbsUpCount": r.get("thumbsUpCount"),
                "appVersion": r.get("reviewCreatedVersion"),
            })

        if token is None:
            break

        time.sleep(sleep_s)

    return pd.DataFrame(all_rows)

In [18]:
for app in APPS:
    bank = app["bank"]
    ios_id = app["ios_app_id"]
    pkg = app["gp_package"]

    print(f"\n Processing: {bank}")

    # App Store
    df_ios = fetch_app_store_reviews_us(ios_id)
    ios_path = os.path.join(DATA_DIR, f"{bank.replace(' ', '_').lower()}__app_store_us.csv")
    df_ios.to_csv(ios_path, index=False)
    print(f" App Store saved: {ios_path} ({len(df_ios)} rows)")

    # Google Play
    df_gp = fetch_google_play_reviews_us(pkg)
    gp_path = os.path.join(DATA_DIR, f"{bank.replace(' ', '_').lower()}__google_play_us.csv")
    df_gp.to_csv(gp_path, index=False)
    print(f" Google Play saved: {gp_path} ({len(df_gp)} rows)")

    # Combined
    df_all = pd.concat([df_ios, df_gp], ignore_index=True)
    combined_path = os.path.join(DATA_DIR, f"{bank.replace(' ', '_').lower()}__combined_us.csv")
    df_all.to_csv(combined_path, index=False)
    print(f"Combined saved: {combined_path}")


 Processing: Chase
 App Store saved: /content/Dataset/data_new_approach/chase__app_store_us.csv (490 rows)
 Google Play saved: /content/Dataset/data_new_approach/chase__google_play_us.csv (600 rows)
Combined saved: /content/Dataset/data_new_approach/chase__combined_us.csv

 Processing: Bank of America
 App Store saved: /content/Dataset/data_new_approach/bank_of_america__app_store_us.csv (490 rows)
 Google Play saved: /content/Dataset/data_new_approach/bank_of_america__google_play_us.csv (600 rows)
Combined saved: /content/Dataset/data_new_approach/bank_of_america__combined_us.csv

 Processing: Wells Fargo
 App Store saved: /content/Dataset/data_new_approach/wells_fargo__app_store_us.csv (490 rows)
 Google Play saved: /content/Dataset/data_new_approach/wells_fargo__google_play_us.csv (600 rows)
Combined saved: /content/Dataset/data_new_approach/wells_fargo__combined_us.csv

 Processing: Citi
 App Store saved: /content/Dataset/data_new_approach/citi__app_store_us.csv (490 rows)
 Google 